# Text Classification with ClassifierDL - part3

**Relevant blogpost:** https://towardsdatascience.com/text-classification-in-spark-nlp-with-bert-and-universal-sentence-encoders-e644d618ca32

# Case Study: Conference Title Classification 

In [5]:
import sparknlp

from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline
import pandas as pd

spark = sparknlp.start(gpu = False) #True) # for GPU training >> sparknlp.start(gpu = True) # for Spark 2.3 =>> sparknlp.start(spark23 = True)

print("Spark NLP version", sparknlp.version())
print("Apache Spark version:", spark.version)

#spark

Spark NLP version 3.3.2
Apache Spark version: 3.1.2


In [1]:
import pandas as pd
df = pd.read_csv('data/title_conference.csv')
df

,Title,Conference
0,Innovation in Database Management: Computer Sc...,VLDB
1,High performance prime field multiplication fo...,ISCAS
2,enchanted scissors: a scissor interface for su...,SIGGRAPH
3,Detection of channel degradation attack by Int...,INFOCOM
4,Pinning a Complex Network through the Betweenn...,ISCAS
...,...,...
2502,A new QR-decomposition based recursive frequen...,ISCAS
2503,CNN Implementation of Spin Filters for Electro...,ISCAS
2504,FaceKit: A Database Interface Design Toolkit.,VLDB
2505,On the trade-off between the number of scrolls...,ISCAS


In [2]:
df.Conference.value_counts()

ISCAS       864
INFOCOM     515
VLDB        423
WWW         379
SIGGRAPH    326
Name: Conference, dtype: int64

In [6]:
trainDataset = spark.read \
      .option("header", True) \
      .csv("data/title_conference.csv")

(trainingData, testData) = trainDataset.randomSplit([0.8, 0.2], seed = 100)
print("Training Dataset Count: " + str(trainingData.count()))
print("Test Dataset Count: " + str(testData.count()))


Training Dataset Count: 2030
Test Dataset Count: 477


In [7]:
document = DocumentAssembler()\
    .setInputCol("Title")\
    .setOutputCol("document")
    
# we can also use sentece detector here if we want to train on and get predictions for each sentence

use = UniversalSentenceEncoder.pretrained()\
    .setInputCols(["document"])\
    .setOutputCol("sentence_embeddings")

# the classes/labels/categories are in category column
classsifierdl = ClassifierDLApproach()\
    .setInputCols(["sentence_embeddings"])\
    .setOutputCol("class")\
    .setLabelColumn("Conference")\
    .setMaxEpochs(20)\
    .setEnableOutputLogs(True)

use_clf_pipeline = Pipeline(
    stages = [
        document,
        use,
        classsifierdl
    ])

tfhub_use download started this may take some time.
Approximate size to download 923.7 MB
[OK!]


In [8]:
#%%time

use_pipelineModel = use_clf_pipeline.fit(trainingData)

# 20 epochs takes around 20 seconds !

In [9]:
preds = use_pipelineModel.transform(testData)

preds.select('Title','Conference',"class.result").show(10, truncate=80)


+--------------------------------------------------------------------------------+----------+----------+
|                                                                           Title|Conference|    result|
+--------------------------------------------------------------------------------+----------+----------+
|                       "From ""Think Like a Vertex"" to ""Think Like a Graph""."|      VLDB|   [ISCAS]|
|                                                  1, 000 Tables Inside the From.|      VLDB|    [VLDB]|
|                             10Gbit/s 2mW inductorless transimpedance amplifier.|     ISCAS|   [ISCAS]|
|3D reconstruction from planar points: A candidate method for authentication o...|     ISCAS|[SIGGRAPH]|
|        7-decades tunable translinear SiGe BiCMOS 3-phase sinusoidal oscillator.|     ISCAS|   [ISCAS]|
|           A 0.18-&micro;m CMOS Squarer Circuit for a Non-Coherent UWB Receiver.|     ISCAS|   [ISCAS]|
|    A 10-Bit 200-MS/s digitally-calibrated pipelined A

In [10]:
# We are going to use sklearn to evalute the results on test dataset
preds_df = preds.select('Conference','Title',"class.result").toPandas()

# Let's explode the array and get the item(s) inside of result column out
preds_df['result'] = preds_df['result'].apply(lambda x : x[0])

from sklearn.metrics import classification_report

print (classification_report(preds_df['result'], preds_df['Conference']))


              precision    recall  f1-score   support

     INFOCOM       0.61      0.78      0.69        77
       ISCAS       0.85      0.79      0.82       190
    SIGGRAPH       0.65      0.67      0.66        49
        VLDB       0.67      0.56      0.61        82
         WWW       0.65      0.67      0.66        79

    accuracy                           0.72       477
   macro avg       0.69      0.70      0.69       477
weighted avg       0.73      0.72      0.72       477



In [11]:
from sklearn.metrics import confusion_matrix, accuracy_score
import numpy as np

preds = pd.DataFrame(confusion_matrix(preds_df['result'], preds_df['Conference']), columns = np.unique(preds_df['Conference']), index =  np.unique(preds_df['Conference']))
preds

,INFOCOM,ISCAS,SIGGRAPH,VLDB,WWW
INFOCOM,60,8,1,2,6
ISCAS,19,151,10,7,3
SIGGRAPH,1,7,33,3,5
VLDB,9,10,2,46,15
WWW,9,1,5,11,53


### Bert Sentence Embeddings

In [12]:
document = DocumentAssembler()\
    .setInputCol("Title")\
    .setOutputCol("document")
    
bert_sent = BertSentenceEmbeddings.pretrained('sent_small_bert_L8_512')\
    .setInputCols(["document"])\
    .setOutputCol("sentence_embeddings")

# the classes/labels/categories are in category column
classsifierdl = ClassifierDLApproach()\
    .setInputCols(["sentence_embeddings"])\
    .setOutputCol("class")\
    .setLabelColumn("Conference")\
    .setMaxEpochs(20)\
    .setLr(0.001)\
    .setBatchSize(8)\
    .setEnableOutputLogs(True)

bert_clf_pipeline = Pipeline(
    stages = [
        document,
        bert_sent,
        classsifierdl
    ])

sent_small_bert_L8_512 download started this may take some time.
Approximate size to download 149.1 MB
[OK!]


In [13]:
classsifierdl.getMaxEpochs()

20

In [14]:
#%%time
#20 epochs 2 mins !

bert_pipelineModel = bert_clf_pipeline.fit(trainingData)


In [15]:
from sklearn.metrics import classification_report

preds = bert_pipelineModel.transform(testData)

preds_df = preds.select('Conference',"class.result").toPandas()

preds_df['result'] = preds_df['result'].apply(lambda x : x[0])

print (classification_report(preds_df['result'], preds_df['Conference']))


              precision    recall  f1-score   support

     INFOCOM       0.72      0.73      0.73        97
       ISCAS       0.89      0.85      0.87       186
    SIGGRAPH       0.69      0.73      0.71        48
        VLDB       0.64      0.59      0.61        75
         WWW       0.63      0.73      0.68        71

    accuracy                           0.75       477
   macro avg       0.72      0.73      0.72       477
weighted avg       0.76      0.75      0.76       477



# Save model and Zip it for Modelshub Upload/Downloads

In [16]:
# Save a Spark NLP pipeline
bert_clf_pipeline.save('data/my_nlp_pipeline')

# cd into saved dir and zip for linux
#! cd /content/my_nlp_pipeline ; zip -r my_nlp_pipeline.zip *